# imports

In [37]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# connection and load

In [38]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

In [39]:
ASSET = 'BTC'
INTERVAL = '4h'

# Load from Gold Layer

In [40]:
query = f"""
    SELECT * 
    FROM gold_crypto_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

print(f"Loading {ASSET} {INTERVAL} from gold_crypto_features...")
df = conn.execute(query).df()
print(f"Loaded: {df.shape}")

Loading BTC 4h from gold_crypto_features...
Loaded: (13274, 52)


# datetime index

In [41]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,date,open,high,low,close,volume,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
0,BTC,Crypto,Bybit,4h,2020-04-27 12:00:00,7713.5,7722.5,7624.0,7669.5,1068.971,...,-0.005721,0.012843,0.461929,7713.5,832.093,7736.0,7666.0,8.198473e+06,NaN,NaN
1,BTC,Crypto,Bybit,4h,2020-04-27 16:00:00,7669.5,7718.5,7642.0,7700.5,588.447,...,0.004034,0.009934,0.764706,7669.5,1068.971,7722.5,7624.0,4.531336e+06,NaN,NaN
2,BTC,Crypto,Bybit,4h,2020-04-27 20:00:00,7700.5,7777.5,7686.0,7772.5,838.893,...,0.009307,0.011772,0.945355,7700.5,588.447,7718.5,7642.0,6.520296e+06,NaN,NaN


In [42]:
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f"Index set to datetime. Date range: {df.index.min()} → {df.index.max()}")

Index set to datetime. Date range: 2020-04-27 12:00:00 → 2026-05-18 16:00:00


In [43]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,open,high,low,close,volume,daily_volatility,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
date,,,,,,,,,,,,,,,,,,,,,
2020-04-27 12:00:00,BTC,Crypto,Bybit,4h,7713.5,7722.5,7624.0,7669.5,1068.971,98.5,...,-0.005721,0.012843,0.461929,7713.5,832.093,7736.0,7666.0,8.198473e+06,NaN,NaN
2020-04-27 16:00:00,BTC,Crypto,Bybit,4h,7669.5,7718.5,7642.0,7700.5,588.447,76.5,...,0.004034,0.009934,0.764706,7669.5,1068.971,7722.5,7624.0,4.531336e+06,NaN,NaN
2020-04-27 20:00:00,BTC,Crypto,Bybit,4h,7700.5,7777.5,7686.0,7772.5,838.893,91.5,...,0.009307,0.011772,0.945355,7700.5,588.447,7718.5,7642.0,6.520296e+06,NaN,NaN


In [44]:
print(f"Shape: {df.shape}")
df.info(verbose=True, show_counts=True)

Shape: (13274, 51)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 13274 entries, 2020-04-27 12:00:00 to 2026-05-18 16:00:00
Data columns (total 51 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   asset_symbol      13274 non-null  object 
 1   asset_class       13274 non-null  object 
 2   exchange          13274 non-null  object 
 3   interval          13274 non-null  object 
 4   open              13274 non-null  float64
 5   high              13274 non-null  float64
 6   low               13274 non-null  float64
 7   close             13274 non-null  float64
 8   volume            13274 non-null  float64
 9   daily_volatility  13274 non-null  float64
 10  sma_7             13274 non-null  float64
 11  sma_30            13274 non-null  float64
 12  rsi_14            13274 non-null  float64
 13  macd              13274 non-null  float64
 14  macd_signal       13274 non-null  float64
 15  macd_histogram    13274 non-null 

# drop metadata cols

In [45]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 47


# check  specific cols (OI, Turnover, Funding Rate)

In [46]:
new_cols = ['turnover', 'open_interest', 'funding_rate']
available_new = [c for c in new_cols if c in df.columns]
missing_new = [c for c in new_cols if c not in df.columns]

print(f"Columns available: {available_new}")
print(f"Columns MISSING:    {missing_new}")

Columns available: ['turnover', 'open_interest', 'funding_rate']
Columns MISSING:    []


#  coverage analysis

In [47]:
for c in available_new:
    coverage = (1 - df[c].isna().mean()) * 100
    print(f"  {c}: {coverage:.1f}% non-null ({df[c].notna().sum():,} / {len(df):,} rows)")

  turnover: 100.0% non-null (13,274 / 13,274 rows)
  open_interest: 0.0% non-null (0 / 13,274 rows)
  funding_rate: 3.0% non-null (399 / 13,274 rows)


# drop funding_rate (unusable ~3% coverage)

In [48]:
if 'funding_rate' in df.columns:
    df = df.drop(columns=['funding_rate'])
    print("Dropped funding_rate — not suitable for model training")
else:
    print("funding_rate not in cols, nothing to drop")

Dropped funding_rate — not suitable for model training


# open interest feature engineering

In [49]:
if 'open_interest' in df.columns:
    df['oi_change_1p']  = df['open_interest'].pct_change(1)
    df['oi_change_5p']  = df['open_interest'].pct_change(5)
    df['oi_change_20p'] = df['open_interest'].pct_change(20)
    
    oi_roll_mean = df['open_interest'].rolling(50).mean()
    oi_roll_std  = df['open_interest'].rolling(50).std()
    df['oi_zscore_50'] = (df['open_interest'] - oi_roll_mean) / oi_roll_std
    
    df = df.drop(columns=['open_interest'])
    
    print("OI features engineered: oi_change_1p, oi_change_5p, oi_change_20p, oi_zscore_50")
    print(f"  oi_change_1p  NaN: {df['oi_change_1p'].isna().sum()} rows")
    print(f"  oi_change_5p  NaN: {df['oi_change_5p'].isna().sum()} rows")
    print(f"  oi_change_20p NaN: {df['oi_change_20p'].isna().sum()} rows")
    print(f"  oi_zscore_50  NaN: {df['oi_zscore_50'].isna().sum()} rows")

OI features engineered: oi_change_1p, oi_change_5p, oi_change_20p, oi_zscore_50
  oi_change_1p  NaN: 13274 rows
  oi_change_5p  NaN: 13274 rows
  oi_change_20p NaN: 13274 rows
  oi_zscore_50  NaN: 13274 rows


C:\Users\Manindra\AppData\Local\Temp\ipykernel_21176\1064927156.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['oi_change_1p']  = df['open_interest'].pct_change(1)
C:\Users\Manindra\AppData\Local\Temp\ipykernel_21176\1064927156.py:3: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['oi_change_5p']  = df['open_interest'].pct_change(5)
C:\Users\Manindra\AppData\Local\Temp\ipykernel_21176\1064927156.py:4: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_chang

# turnover feature engineering

In [50]:
if 'turnover' in df.columns:
    df['turnover_ratio'] = df['turnover'] / df['volume']
    df['turnover_change_1p'] = df['turnover'].pct_change(1)
    df['turnover_change_5p'] = df['turnover'].pct_change(5)
    
    tv_roll_mean = df['turnover_ratio'].rolling(50).mean()
    tv_roll_std  = df['turnover_ratio'].rolling(50).std()
    df['turnover_ratio_zscore_50'] = (df['turnover_ratio'] - tv_roll_mean) / tv_roll_std
    
    df = df.drop(columns=['turnover'])
    
    print("Turnover features engineered: turnover_ratio, turnover_change_1p, turnover_change_5p, turnover_ratio_zscore_50")
    print(f"  turnover_ratio          NaN: {df['turnover_ratio'].isna().sum()} rows")
    print(f"  turnover_change_1p      NaN: {df['turnover_change_1p'].isna().sum()} rows")
    print(f"  turnover_change_5p      NaN: {df['turnover_change_5p'].isna().sum()} rows")
    print(f"  turnover_ratio_zscore_50 NaN: {df['turnover_ratio_zscore_50'].isna().sum()} rows")

Turnover features engineered: turnover_ratio, turnover_change_1p, turnover_change_5p, turnover_ratio_zscore_50
  turnover_ratio          NaN: 0 rows
  turnover_change_1p      NaN: 1 rows
  turnover_change_5p      NaN: 5 rows
  turnover_ratio_zscore_50 NaN: 49 rows


# native target 4h (1-period forward return!)

In [51]:
df['target_4h'] = df['close'].pct_change(1).shift(-1)

# drop target nan the last row 

In [52]:
df = df.dropna(subset=['target_4h'])
print(f"Shape after dropping target NaN: {df.shape}")

Shape after dropping target NaN: (13273, 53)


In [53]:
df[['close', 'target_4h']].tail(6)

,close,target_4h
date,,
2026-05-17 16:00:00,78370.0,-0.012168
2026-05-17 20:00:00,77416.4,-0.006665
2026-05-18 00:00:00,76900.4,0.001856
2026-05-18 04:00:00,77043.1,0.002841
2026-05-18 08:00:00,77262.0,-0.011400
2026-05-18 12:00:00,76381.2,-0.000055


# drop remaining nan

In [54]:
df = df.dropna()
print(f"Data shape after dropping all NaN: {df.shape}")
print(f"Total features: {len(df.columns) - 2}  (+ target_4h + target_direction)")

Data shape after dropping all NaN: (0, 53)
Total features: 51  (+ target_4h + target_direction)


# stationarity transform: MAs → distance from price (%)

In [55]:
df['sma_7_dist']   = df['close'] / df['sma_7']   - 1
df['sma_30_dist']  = df['close'] / df['sma_30']  - 1
df['sma_50_dist']  = df['close'] / df['sma_50']  - 1
df['sma_100_dist'] = df['close'] / df['sma_100'] - 1
df['sma_200_dist'] = df['close'] / df['sma_200'] - 1
df['ema_12_dist']  = df['close'] / df['ema_12']  - 1
df['ema_26_dist']  = df['close'] / df['ema_26']  - 1
df['ema_50_dist']  = df['close'] / df['ema_50']  - 1
df['ema_200_dist'] = df['close'] / df['ema_200'] - 1
df['vwap_dist']    = df['close'] / df['vwap']    - 1

print("MA distances created (close/MA - 1)")

MA distances created (close/MA - 1)


# stationarity transform: MACD/ATR/Vol → % of price

In [56]:
df['macd_pct']      = df['macd']            / df['close']
df['macd_sig_pct']  = df['macd_signal']     / df['close']
df['macd_hist_pct'] = df['macd_histogram']  / df['close']
df['atr_pct']       = df['atr_14']          / df['close']
df['volatility_pct']= df['daily_volatility'] / df['close']

print("MACD/ATR/Vol scaled as % of price")

MACD/ATR/Vol scaled as % of price


# drop non stationary cols

In [57]:
non_stationary_cols = [
    'open', 'high', 'low', 'close', 
    'sma_7', 'sma_30', 'sma_50', 'sma_100', 'sma_200',
    'ema_12', 'ema_26', 'ema_50', 'ema_200', 
    'vwap', 'prev_close', 'prev_high', 'prev_low',
    'macd', 'macd_signal', 'macd_histogram', 
    'atr_14', 'daily_volatility',
    'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 
    'volume', 'prev_volume', 'volume_sma_20', 
    'obv' 
]

df = df.drop(columns=non_stationary_cols, errors='ignore')
print(f"Dropped {len(non_stationary_cols)} non-stationary columns")
print(f"Columns remaining: {len(df.columns)}")

Dropped 30 non-stationary columns
Columns remaining: 38
